In [1]:
# ============================================================
# MVTec 4-class counterfactual configuration generation
# ============================================================

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor

import dice_ml
from dice_ml import Dice

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)

SEED = 42
rng = np.random.default_rng(SEED)

CWD = Path.cwd()
print("Current working directory:")
print(CWD)

DATA_FILENAME = "mvtec_4class_patchcore_surrogate_dataset.csv"
MODEL_FILENAME = "mvtec_surrogate_catboost.cbm"
METADATA_FILENAME = "mvtec_surrogate_metadata.json"

candidate_data_paths = [
    CWD / "surrogate_ready_dataset" / DATA_FILENAME,
    CWD / "hpo_mvtec_4class" / "surrogate_ready_dataset" / DATA_FILENAME,
    CWD.parent / "surrogate_ready_dataset" / DATA_FILENAME,
    CWD.parent / "hpo_mvtec_4class" / "surrogate_ready_dataset" / DATA_FILENAME,
]

for base in [CWD, CWD.parent]:
    try:
        candidate_data_paths.extend(base.rglob(DATA_FILENAME))
    except Exception:
        pass

seen = set()
candidate_data_paths_unique = []
for p in candidate_data_paths:
    p = p.resolve()
    if p not in seen:
        seen.add(p)
        candidate_data_paths_unique.append(p)

existing_data_paths = []

print("\nCandidate dataset paths:")
for p in candidate_data_paths_unique:
    exists = p.exists()
    print(f"- exists={exists:<5} | {p}")
    if exists:
        existing_data_paths.append(p)

if not existing_data_paths:
    raise FileNotFoundError(
        "Could not find the MVTec surrogate-ready dataset. "
        "Run the MVTec preprocessing notebook first."
    )

CSV_PATH = existing_data_paths[0]
ROOT = CSV_PATH.parent.parent

ART_DIR = ROOT / "surrogate_pkl_cfs_metadata"
MODEL_PATH = ART_DIR / MODEL_FILENAME
METADATA_PATH = ART_DIR / METADATA_FILENAME

EXPORT_DIR = ART_DIR / "mvtec_counterfactuals"
FIG_DIR = ROOT / "figures_counterfactuals"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("\nResolved paths:")
print("ROOT:", ROOT)
print("CSV_PATH:", CSV_PATH)
print("MODEL_PATH:", MODEL_PATH)
print("METADATA_PATH:", METADATA_PATH)
print("EXPORT_DIR:", EXPORT_DIR)

print("\nFile checks:")
print("Dataset exists:", CSV_PATH.exists())
print("Model exists:", MODEL_PATH.exists())
print("Metadata exists:", METADATA_PATH.exists())

Current working directory:
/home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class

Candidate dataset paths:
- exists=1     | /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv
- exists=0     | /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/hpo_mvtec_4class/surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv
- exists=0     | /home/jovyan/fast/x_peer_pdt_hyperoptxai/surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv

Resolved paths:
ROOT: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class
CSV_PATH: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv
MODEL_PATH: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_surrogate_catboost.cbm
METADATA_PATH: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_surroga

In [2]:
# ============================================================
# 1) Configuration schema
# ============================================================

TARGET = "value"

if METADATA_PATH.exists():
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    FEATURES = metadata["features"]
    CATEGORICAL = metadata["categorical_features"]
    NUMERIC = metadata["numeric_features"]

    print("Loaded schema from metadata.")
else:
    FEATURES = [
        "params_backbone",
        "params_batch_size",
        "params_center_crop_key",
        "params_coreset_sampling_ratio",
        "params_image_size_key",
        "params_layers_key",
        "params_max_patches_per_image",
        "params_num_neighbors",
        "params_pre_trained",
        "params_reduction",
        "params_soft_corruption_level",
        "params_soft_review_budget",
        "params_soft_train_fraction",
    ]

    CATEGORICAL = [
        "params_backbone",
        "params_batch_size",
        "params_center_crop_key",
        "params_image_size_key",
        "params_layers_key",
        "params_max_patches_per_image",
        "params_pre_trained",
        "params_reduction",
        "params_soft_corruption_level",
    ]

    NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]

    print("Metadata not found. Using fallback schema.")

SOFT_TRAIN_FRACTION_RANGE = (0.20, 1.00)
SOFT_CORRUPTION_LEVELS = ["none", "mild", "strong"]
SOFT_REVIEW_BUDGETS = [i / 1000 for i in range(5, 501, 5)]  # 0.005..0.500

print("Target:", TARGET)
print("Features:", len(FEATURES))
print("Categorical:", CATEGORICAL)
print("Numeric:", NUMERIC)

Loaded schema from metadata.
Target: value
Features: 13
Categorical: ['params_backbone', 'params_batch_size', 'params_center_crop_key', 'params_image_size_key', 'params_layers_key', 'params_max_patches_per_image', 'params_pre_trained', 'params_reduction', 'params_soft_corruption_level']
Numeric: ['params_coreset_sampling_ratio', 'params_num_neighbors', 'params_soft_review_budget', 'params_soft_train_fraction']


In [3]:
# ============================================================
# 2) Helper functions
# ============================================================

def pick(prompt, options):
    while True:
        print("\n" + prompt)
        for i, opt in enumerate(options, start=1):
            print(f"  {i}) {opt}")
        s = input("Choose: ").strip()
        if s.isdigit():
            k = int(s)
            if 1 <= k <= len(options):
                return k - 1
        print("Invalid choice. Please type a number from the list.")


def ask_float(prompt, lo, hi, step=None):
    while True:
        s = input(f"{prompt} [{lo}..{hi}]: ").strip().replace(",", ".")
        try:
            x = float(s)

            if x < lo or x > hi:
                print("Out of range.")
                continue

            if step is not None:
                x = round((x - lo) / step) * step + lo
                x = max(lo, min(hi, x))
                x = float(f"{x:.6f}")

            return x

        except ValueError:
            print("Invalid number. Try again.")


def snap_to_grid(x, grid):
    return min(grid, key=lambda b: abs(b - x))


def snap_budget(x):
    return snap_to_grid(x, SOFT_REVIEW_BUDGETS)


def show_row(row, cols):
    return ", ".join([f"{c}={row[c]}" for c in cols])


def predict_surrogate(model, X):
    Xp = X[FEATURES].copy()

    for c in CATEGORICAL:
        Xp[c] = Xp[c].astype(str)

    for c in NUMERIC:
        Xp[c] = pd.to_numeric(Xp[c], errors="coerce")

    return np.asarray(model.predict(Xp), dtype=float)


def domain_for_feature(df_ref, feat):
    if feat in CATEGORICAL:
        return sorted(df_ref[feat].dropna().astype(str).unique().tolist())

    col = pd.to_numeric(df_ref[feat], errors="coerce")
    return [float(np.nanmin(col)), float(np.nanmax(col))]


def safe_range(df_ref, lo, hi, feat):
    try:
        lo = float(lo)
        hi = float(hi)

        if lo > hi:
            return None

        dom_lo, dom_hi = domain_for_feature(df_ref, feat)

        lo = max(dom_lo, lo)
        hi = min(dom_hi, hi)

        if lo > hi:
            return None

        return [float(lo), float(hi)]

    except Exception:
        return None


def safe_allowed(df_ref, allowed, feat):
    try:
        dom = set(map(str, domain_for_feature(df_ref, feat)))
        allowed = [str(a) for a in allowed]
        allowed = [a for a in allowed if a in dom]

        return sorted(set(allowed)) if allowed else None

    except Exception:
        return None


def mixed_distance(row, factual_row, df_ref):
    distance = 0.0

    for c in NUMERIC:
        x = float(row[c])
        x0 = float(factual_row[c])
        dom_lo, dom_hi = domain_for_feature(df_ref, c)
        span = max(dom_hi - dom_lo, 1e-9)
        distance += abs(x - x0) / span

    for c in CATEGORICAL:
        distance += int(str(row[c]) != str(factual_row[c]))

    return float(distance)


def check_soft_constraints(row, soft_constraints):
    violations = []

    for feat, spec in soft_constraints.items():
        mode = spec.get("mode", "free")

        if mode == "free":
            continue

        if feat in CATEGORICAL:
            val = str(row[feat])

            if mode == "fixed":
                expected = str(spec.get("value"))
                if val != expected:
                    violations.append(f"{feat} != {expected}")

            elif mode == "allowed":
                allowed = set(map(str, spec.get("allowed", [])))
                if val not in allowed:
                    violations.append(f"{feat} not in {sorted(allowed)}")

        else:
            val = float(row[feat])

            if mode == "fixed":
                expected = float(spec.get("value"))
                if abs(val - expected) > 1e-9:
                    violations.append(f"{feat} != {expected}")

            elif mode == "range":
                lo, hi = spec.get("bounds", [None, None])
                lo = float(lo)
                hi = float(hi)
                if val < lo or val > hi:
                    violations.append(f"{feat} not in [{lo}, {hi}]")

    return violations


def rank_counterfactuals(cf_df, factual_row, df_ref, mode):
    cf_df = cf_df.copy()

    cf_df["distance_to_factual"] = cf_df.apply(
        lambda r: mixed_distance(r, factual_row, df_ref),
        axis=1,
    )

    if mode == "best":
        sort_cols = ["n_violations", TARGET, "distance_to_factual"]
        ascending = [True, False, True]

    elif mode == "cheapest":
        sort_cols = ["n_violations", "params_soft_review_budget", TARGET, "distance_to_factual"]
        ascending = [True, True, False, True]

    else:  # balanced
        sort_cols = ["n_violations", TARGET, "params_soft_review_budget", "distance_to_factual"]
        ascending = [True, False, True, True]

    return cf_df.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)

In [4]:
# ============================================================
# 3) Load MVTec surrogate-ready dataset and CatBoost surrogate
# ============================================================

assert CSV_PATH.is_file(), f"Missing dataset: {CSV_PATH}"
assert MODEL_PATH.is_file(), f"Missing surrogate model: {MODEL_PATH}"

df = pd.read_csv(CSV_PATH)

missing = [c for c in [TARGET] + FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in dataset: {missing}")

df = df[[TARGET] + FEATURES].copy()

for c in CATEGORICAL:
    df[c] = df[c].astype(str)

for c in NUMERIC + [TARGET]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

before = len(df)
df = df.dropna(subset=[TARGET] + FEATURES).reset_index(drop=True)
after = len(df)

cb = CatBoostRegressor()
cb.load_model(str(MODEL_PATH))

print("Loaded dataset:", CSV_PATH)
print("Dataset shape:", df.shape)
print("Dropped rows after cleaning:", before - after)
print("Loaded surrogate:", MODEL_PATH)

print("\nTarget summary:")
display(df[TARGET].describe())

# Quick prediction sanity check
sample_pred = predict_surrogate(cb, df.head(5))
print("\nPrediction sanity check:")
print(sample_pred)

Loaded dataset: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_ready_dataset/mvtec_4class_patchcore_surrogate_dataset.csv
Dataset shape: (9976, 14)
Dropped rows after cleaning: 0
Loaded surrogate: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_surrogate_catboost.cbm

Target summary:


count    9976.000000
mean        0.524907
std         0.228218
min         0.015625
25%         0.347519
50%         0.562076
75%         0.732452
max         0.820249
Name: value, dtype: float64


Prediction sanity check:
[0.56857286 0.56857286 0.29031693 0.35167161 0.56857286]


In [5]:
# ============================================================
# 4) Pick factual configuration
# ============================================================

show_cols = [
    "params_backbone",
    "params_image_size_key",
    "params_layers_key",
    "params_soft_corruption_level",
    "params_soft_review_budget",
    "params_soft_train_fraction",
]

# Use stable candidates from different regions of the HPO log
candidate_indices = []

quantiles = [0.50, 0.70, 0.85, 0.95]
for q in quantiles:
    target_value = df[TARGET].quantile(q)
    idx = (df[TARGET] - target_value).abs().idxmin()
    candidate_indices.append(idx)

# Add the best configuration
candidate_indices.append(df[TARGET].idxmax())

# Remove duplicates while preserving order
candidate_indices = list(dict.fromkeys(candidate_indices))

print("\n=== Pick a factual configuration / starting point ===\n")

cands = []

for i, idx in enumerate(candidate_indices, start=1):
    row = df.loc[idx, FEATURES + [TARGET]].copy()
    pred = float(predict_surrogate(cb, pd.DataFrame([row[FEATURES]]))[0])

    print(
        f"[{i}] index={idx} | observed={row[TARGET]:.6f} | "
        f"surrogate={pred:.6f} | " + show_row(row, show_cols)
    )

    cands.append((idx, row, pred))

choice = pick(
    "Select factual configuration:",
    [f"Factual {i}" for i in range(1, len(cands) + 1)],
)

selected_idx, selected_row, selected_pred = cands[choice]

x_factual = selected_row[FEATURES].copy()
y_factual_true = float(selected_row[TARGET])
y_factual_sur = float(selected_pred)

x_factual_df = pd.DataFrame([x_factual.values], columns=FEATURES)

for c in CATEGORICAL:
    x_factual_df[c] = x_factual_df[c].astype(str)

for c in NUMERIC:
    x_factual_df[c] = pd.to_numeric(x_factual_df[c], errors="coerce")

print("\nSelected factual index:", int(selected_idx))
print(f"Observed value: {y_factual_true:.6f}")
print(f"Surrogate prediction: {y_factual_sur:.6f}")

display(x_factual_df)


=== Pick a factual configuration / starting point ===

[1] index=565 | observed=0.562076 | surrogate=0.552129 | params_backbone=resnet18, params_image_size_key=224, params_layers_key=l2, params_soft_corruption_level=mild, params_soft_review_budget=0.27, params_soft_train_fraction=0.75
[2] index=677 | observed=0.693407 | surrogate=0.715547 | params_backbone=resnet34, params_image_size_key=256, params_layers_key=l1_l2_l3, params_soft_corruption_level=none, params_soft_review_budget=0.42, params_soft_train_fraction=0.78
[3] index=4800 | observed=0.800621 | surrogate=0.796102 | params_backbone=wide_resnet50_2, params_image_size_key=256, params_layers_key=l2_l3, params_soft_corruption_level=none, params_soft_review_budget=0.49, params_soft_train_fraction=0.89
[4] index=1628 | observed=0.808344 | surrogate=0.807137 | params_backbone=wide_resnet50_2, params_image_size_key=256, params_layers_key=l2, params_soft_corruption_level=none, params_soft_review_budget=0.495, params_soft_train_fraction

Choose:  1



Selected factual index: 565
Observed value: 0.562076
Surrogate prediction: 0.552129


,params_backbone,params_batch_size,params_center_crop_key,params_image_size_key,params_layers_key,params_max_patches_per_image,params_pre_trained,params_reduction,params_soft_corruption_level,params_coreset_sampling_ratio,params_num_neighbors,params_soft_review_budget,params_soft_train_fraction
0,resnet18,16,0.875,224,l2,512,True,mean,mild,0.029,9,0.27,0.75


In [6]:
# ============================================================
# 5) User objective and soft constraints
# ============================================================

print("\n=== Soft knobs ===")
print("params_soft_train_fraction  = training data fraction / data effort")
print("params_soft_review_budget   = review effort at deployment")
print("params_soft_corruption_level = data-quality / corruption assumption")
print("Performance value           = surrogate-predicted validation F1\n")

goal_type_idx = pick(
    "Performance goal:",
    [
        "Reach at least a minimum F1",
        "Improve by a delta over the factual",
        "Stay within a target band [L, U]",
    ],
)

objective = {"type": None}

if goal_type_idx == 0:
    t = ask_float("Enter target minimum F1", 0.0, 1.0)
    objective.update({"type": "target_min", "target_min": t})

elif goal_type_idx == 1:
    d = ask_float("Enter delta improvement", 0.0, 1.0)
    objective.update({"type": "delta_improve", "delta": d})

else:
    L = ask_float("Enter target band lower bound", 0.0, 1.0)
    U = ask_float("Enter target band upper bound", L, 1.0)
    objective.update({"type": "target_band", "target_band": [L, U]})


soft_constraints = {}

# ------------------------------------------------------------
# Training fraction
# ------------------------------------------------------------

mode_idx = pick(
    "Constraint for training fraction:",
    ["Free", "Fix to a value", "Restrict to a range"],
)

if mode_idx == 0:
    soft_constraints["params_soft_train_fraction"] = {"mode": "free"}

elif mode_idx == 1:
    v = ask_float(
        "Fix training fraction to",
        SOFT_TRAIN_FRACTION_RANGE[0],
        SOFT_TRAIN_FRACTION_RANGE[1],
        step=0.01,
    )
    soft_constraints["params_soft_train_fraction"] = {
        "mode": "fixed",
        "value": float(v),
    }

else:
    lo = ask_float(
        "Training fraction lower bound",
        SOFT_TRAIN_FRACTION_RANGE[0],
        SOFT_TRAIN_FRACTION_RANGE[1],
        step=0.01,
    )
    hi = ask_float(
        "Training fraction upper bound",
        lo,
        SOFT_TRAIN_FRACTION_RANGE[1],
        step=0.01,
    )
    soft_constraints["params_soft_train_fraction"] = {
        "mode": "range",
        "bounds": [float(lo), float(hi)],
    }


# ------------------------------------------------------------
# Review budget
# ------------------------------------------------------------

mode_idx = pick(
    "Constraint for review budget:",
    ["Free", "Fix to a value", "Restrict to a range"],
)

if mode_idx == 0:
    soft_constraints["params_soft_review_budget"] = {"mode": "free"}

elif mode_idx == 1:
    v = ask_float(
        "Fix review budget to",
        min(SOFT_REVIEW_BUDGETS),
        max(SOFT_REVIEW_BUDGETS),
        step=0.005,
    )
    v = snap_budget(v)
    soft_constraints["params_soft_review_budget"] = {
        "mode": "fixed",
        "value": float(v),
    }

else:
    lo = ask_float(
        "Review budget lower bound",
        min(SOFT_REVIEW_BUDGETS),
        max(SOFT_REVIEW_BUDGETS),
        step=0.005,
    )
    hi = ask_float(
        "Review budget upper bound",
        lo,
        max(SOFT_REVIEW_BUDGETS),
        step=0.005,
    )
    lo = snap_budget(lo)
    hi = snap_budget(hi)

    soft_constraints["params_soft_review_budget"] = {
        "mode": "range",
        "bounds": [float(lo), float(hi)],
    }


# ------------------------------------------------------------
# Corruption level
# ------------------------------------------------------------

mode_idx = pick(
    "Constraint for corruption level:",
    ["Free", "Fix to factual value", "Choose allowed levels"],
)

if mode_idx == 0:
    soft_constraints["params_soft_corruption_level"] = {"mode": "free"}

elif mode_idx == 1:
    soft_constraints["params_soft_corruption_level"] = {
        "mode": "fixed",
        "value": str(x_factual["params_soft_corruption_level"]),
    }

else:
    print("\nSelect allowed corruption levels. Example: 1 2")
    for i, lvl in enumerate(SOFT_CORRUPTION_LEVELS, start=1):
        print(f"  {i}) {lvl}")

    while True:
        s = input("Allowed levels: ").strip()
        parts = [p for p in s.split() if p.isdigit()]
        chosen = sorted(set(int(p) for p in parts if 1 <= int(p) <= len(SOFT_CORRUPTION_LEVELS)))

        if chosen:
            allowed = [SOFT_CORRUPTION_LEVELS[i - 1] for i in chosen]
            break

        print("Invalid. Example: 1 2 3")

    soft_constraints["params_soft_corruption_level"] = {
        "mode": "allowed",
        "allowed": allowed,
    }


k = int(["3", "5", "8"][pick("How many counterfactual options to return?", ["3", "5", "8"])])

rank_mode = ["best", "cheapest", "balanced"][
    pick("Selection mode:", ["best performance", "cheapest effort", "balanced"])
]

query = {
    "dataset": "mvtec_4class",
    "method": "dice_random",
    "factual": {
        "index": int(selected_idx),
        "x": x_factual.to_dict(),
        "value_observed": y_factual_true,
        "value_surrogate": y_factual_sur,
    },
    "objective": objective,
    "soft_constraints": soft_constraints,
    "selection": {
        "k": int(k),
        "mode": rank_mode,
    },
}

print("\n=== Query ===")
print(json.dumps(query, indent=2))


=== Soft knobs ===
params_soft_train_fraction  = training data fraction / data effort
params_soft_review_budget   = review effort at deployment
params_soft_corruption_level = data-quality / corruption assumption
Performance value           = surrogate-predicted validation F1


Performance goal:
  1) Reach at least a minimum F1
  2) Improve by a delta over the factual
  3) Stay within a target band [L, U]


Choose:  1
Enter target minimum F1 [0.0..1.0]:  0.7



Constraint for training fraction:
  1) Free
  2) Fix to a value
  3) Restrict to a range


Choose:  3
Training fraction lower bound [0.2..1.0]:  0.2
Training fraction upper bound [0.2..1.0]:  1



Constraint for review budget:
  1) Free
  2) Fix to a value
  3) Restrict to a range


...


How many counterfactual options to return?
  1) 3
  2) 5
  3) 8


Choose:  3



Selection mode:
  1) best performance
  2) cheapest effort
  3) balanced


Choose:  3



=== Query ===
{
  "dataset": "mvtec_4class",
  "method": "dice_random",
  "factual": {
    "index": 565,
    "x": {
      "params_backbone": "resnet18",
      "params_batch_size": "16",
      "params_center_crop_key": "0.875",
      "params_image_size_key": "224",
      "params_layers_key": "l2",
      "params_max_patches_per_image": "512",
      "params_pre_trained": "True",
      "params_reduction": "mean",
      "params_soft_corruption_level": "mild",
      "params_coreset_sampling_ratio": 0.029,
      "params_num_neighbors": 9,
      "params_soft_review_budget": 0.27,
      "params_soft_train_fraction": 0.75
    },
    "value_observed": 0.5620762862970905,
    "value_surrogate": 0.5521290187317024
  },
  "objective": {
    "type": "target_min",
    "target_min": 0.7
  },
  "soft_constraints": {
    "params_soft_train_fraction": {
      "mode": "range",
      "bounds": [
        0.2,
        1.0
      ]
    },
    "params_soft_review_budget": {
      "mode": "free"
    },
    "para

In [7]:
# ============================================================
# 6) Objective interval and permitted ranges
# ============================================================

obj = query["objective"]
factual_base = float(query["factual"]["value_surrogate"])

if obj["type"] == "target_min":
    L, U = float(obj["target_min"]), 1.0

elif obj["type"] == "delta_improve":
    L, U = factual_base + float(obj["delta"]), 1.0

elif obj["type"] == "target_band":
    L, U = map(float, obj["target_band"])

else:
    raise ValueError(f"Unknown objective type: {obj['type']}")

desired_range = [max(0.0, L), min(1.0, U)]

print("Factual surrogate value:", factual_base)
print("Desired range:", desired_range)

x0_dice = x_factual_df[FEATURES].copy()

def factual_value(feat):
    v = x0_dice.iloc[0][feat]
    return str(v) if feat in CATEGORICAL else float(v)

permitted_range = {}

for feat, spec in query["soft_constraints"].items():
    mode = spec.get("mode", "free")

    if mode == "free":
        permitted_range[feat] = domain_for_feature(df, feat)

    elif mode == "fixed":
        if feat in CATEGORICAL:
            v = str(spec.get("value", factual_value(feat)))
            ok = safe_allowed(df, [v], feat)
            if ok is None:
                ok = safe_allowed(df, [factual_value(feat)], feat)
            permitted_range[feat] = ok
        else:
            v = spec.get("value", factual_value(feat))
            r = safe_range(df, v, v, feat)
            if r is None:
                r = safe_range(df, factual_value(feat), factual_value(feat), feat)
            permitted_range[feat] = r

    elif mode == "range":
        if feat in CATEGORICAL:
            permitted_range[feat] = safe_allowed(df, [factual_value(feat)], feat)
        else:
            bounds = spec.get("bounds", None)
            if not bounds or len(bounds) != 2:
                permitted_range[feat] = safe_range(df, factual_value(feat), factual_value(feat), feat)
            else:
                r = safe_range(df, bounds[0], bounds[1], feat)
                if r is None:
                    r = safe_range(df, factual_value(feat), factual_value(feat), feat)
                permitted_range[feat] = r

    elif mode == "allowed":
        if feat not in CATEGORICAL:
            permitted_range[feat] = safe_range(df, factual_value(feat), factual_value(feat), feat)
        else:
            allowed = spec.get("allowed", None)
            ok = safe_allowed(df, allowed, feat) if allowed is not None else None
            if ok is None:
                ok = safe_allowed(df, [factual_value(feat)], feat)
            permitted_range[feat] = ok

    else:
        if feat in CATEGORICAL:
            permitted_range[feat] = safe_allowed(df, [factual_value(feat)], feat)
        else:
            permitted_range[feat] = safe_range(df, factual_value(feat), factual_value(feat), feat)

permitted_range = {k: v for k, v in permitted_range.items() if v is not None}

print("\nPermitted ranges injected into DiCE:")
for k_, v_ in permitted_range.items():
    print(f"- {k_}: {v_}")

Factual surrogate value: 0.5521290187317024
Desired range: [0.7, 1.0]

Permitted ranges injected into DiCE:
- params_soft_train_fraction: [0.2, 1.0]
- params_soft_review_budget: [0.005, 0.5]
- params_soft_corruption_level: ['mild', 'none', 'strong']


In [8]:
# ============================================================
# 7) Common CF validation and ranking for all methods
# ============================================================

def align_candidates_for_surrogate(candidates, feature_cols, categorical_cols, numeric_cols):
    """
    Prepare generated CF configurations for fresh surrogate prediction.
    Keeps only the surrogate feature columns and enforces the expected schema.
    """
    x = candidates.copy()

    extra_cols = [c for c in x.columns if c not in feature_cols]
    if extra_cols:
        x = x.drop(columns=extra_cols, errors="ignore")

    x = x.reindex(columns=feature_cols)

    for c in categorical_cols:
        if c in x.columns:
            x[c] = x[c].astype(str)

    for c in numeric_cols:
        if c in x.columns:
            x[c] = pd.to_numeric(x[c], errors="coerce")

    return x


def fresh_surrogate_predict(model, candidates):
    """
    Fresh surrogate prediction for generated CF configurations.
    """
    x = align_candidates_for_surrogate(
        candidates=candidates,
        feature_cols=FEATURES,
        categorical_cols=CATEGORICAL,
        numeric_cols=NUMERIC,
    )
    return predict_surrogate(model, x[FEATURES])


def safe_mean(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return np.nan
    return float(values.mean())


method_runtime = {}


def prepare_candidate_table(
    candidates,
    method_name,
    factual_row,
    df_ref,
    desired_range,
    soft_constraints,
    model,
):
    """
    Convert candidates from any CF method into the same table format.

    Important columns:
    - cf_method_score: score returned by the CF generator if available.
      For DiCE, this is the DiCE-returned target column.
      For CFNOW, this is the observed HPO-log target if preserved.
      For ACFX, this is usually NaN because the adapter does not return F1 directly.
    - surrogate_pred: fresh CatBoost surrogate prediction for the generated configuration.
    - TARGET/value: kept equal to surrogate_pred for validation/ranking compatibility.
    """

    if candidates is None or len(candidates) == 0:
        return pd.DataFrame()

    raw = candidates.copy().reset_index(drop=True)

    # Preserve method-generated score before dropping non-feature columns.
    if "cf_method_score" in raw.columns:
        cf_method_score = pd.to_numeric(raw["cf_method_score"], errors="coerce")
    elif TARGET in raw.columns:
        cf_method_score = pd.to_numeric(raw[TARGET], errors="coerce")
    else:
        cf_method_score = pd.Series(np.nan, index=raw.index)

    out = align_candidates_for_surrogate(
        candidates=raw,
        feature_cols=FEATURES,
        categorical_cols=CATEGORICAL,
        numeric_cols=NUMERIC,
    )

    out["cf_method_score"] = cf_method_score.to_numpy()

    out = out.dropna(subset=FEATURES).reset_index(drop=True)

    if out.empty:
        return out

    # Fresh surrogate score.
    out["surrogate_pred"] = predict_surrogate(model, out[FEATURES])

    # Keep TARGET/value as surrogate score for ranking and objective checking.
    out[TARGET] = out["surrogate_pred"]
    out["surrogate_target_value"] = out["surrogate_pred"]

    out["delta_surrogate_minus_method_score"] = (
        out["surrogate_pred"] - out["cf_method_score"]
    )

    # Backward-compatible name, but now it means:
    # fresh surrogate prediction minus preserved CF-method score.
    out["gap_pred_minus_cfvalue"] = out["delta_surrogate_minus_method_score"]

    L, U = desired_range

    out["objective_ok"] = (
        (out["surrogate_pred"] >= L)
        & (out["surrogate_pred"] <= U)
    )

    soft_violations = []
    for _, row in out.iterrows():
        soft_violations.append(check_soft_constraints(row, soft_constraints))

    out["soft_violations"] = soft_violations
    out["n_violations"] = out["soft_violations"].apply(len)
    out["soft_constraints_ok"] = out["n_violations"] == 0
    out["strictly_valid"] = out["objective_ok"] & out["soft_constraints_ok"]

    out["distance_to_factual"] = out.apply(
        lambda r: mixed_distance(r, factual_row, df_ref),
        axis=1,
    )

    out = out[out["distance_to_factual"] > 1e-12].copy()
    out = out.drop_duplicates(subset=FEATURES).reset_index(drop=True)

    out["method"] = method_name

    out = rank_counterfactuals(
        out,
        factual_row=factual_row,
        df_ref=df_ref,
        mode=query["selection"]["mode"],
    )

    out["method_rank"] = np.arange(len(out))

    return out.reset_index(drop=True)


def select_top_k(common_df, k):
    """
    Select k candidates from a common-format CF table.
    Prefer strictly valid candidates; if fewer than k exist, include near-valid ones.
    """
    if common_df is None or common_df.empty:
        return pd.DataFrame()

    strict = common_df[common_df["strictly_valid"]].copy()

    if len(strict) >= k:
        return strict.head(k).reset_index(drop=True)

    fallback = common_df.head(k).copy().reset_index(drop=True)

    print(
        f"Only {len(strict)} strictly valid candidates available; "
        f"returning {len(fallback)} ranked candidates including near-valid options."
    )

    return fallback


def summarize_method(common_df, method_name, runtime_sec):
    if common_df is None or common_df.empty:
        return {
            "method": method_name,
            "raw_candidates": 0,
            "returned": 0,
            "strictly_valid": 0,
            "strictly_valid_pct": 0.0,
            "mean_cf_method_score": np.nan,
            "mean_surrogate_f1": np.nan,
            "mean_delta_surrogate_minus_method_score": np.nan,
            "mean_distance": np.nan,
            "mean_violations": np.nan,
            "generation_sec": runtime_sec,
        }

    return {
        "method": method_name,
        "raw_candidates": int(len(common_df)),
        "returned": int(len(common_df)),
        "strictly_valid": int(common_df["strictly_valid"].sum()),
        "strictly_valid_pct": float(100 * common_df["strictly_valid"].mean()),
        "mean_cf_method_score": safe_mean(common_df["cf_method_score"]),
        "mean_surrogate_f1": safe_mean(common_df["surrogate_pred"]),
        "mean_delta_surrogate_minus_method_score": safe_mean(
            common_df["delta_surrogate_minus_method_score"]
        ),
        "mean_distance": safe_mean(common_df["distance_to_factual"]),
        "mean_violations": safe_mean(common_df["n_violations"]),
        "generation_sec": float(runtime_sec) if runtime_sec is not None and not pd.isna(runtime_sec) else np.nan,
    }


K_RETURN = int(query["selection"]["k"])
print("Common CF comparison setup ready.")
print("K_RETURN:", K_RETURN)
print("Desired range:", desired_range)

Common CF comparison setup ready.
K_RETURN: 8
Desired range: [0.7, 1.0]


In [9]:
# ============================================================
# 9) Method 1: DiCE
# ============================================================

df_dice = df[FEATURES + [TARGET]].copy()

for c in CATEGORICAL:
    df_dice[c] = df_dice[c].astype(str)

CONTINUOUS = [c for c in FEATURES if c not in CATEGORICAL]

data_dice = dice_ml.Data(
    dataframe=df_dice,
    continuous_features=CONTINUOUS,
    outcome_name=TARGET,
)

model_dice = dice_ml.Model(
    model=cb,
    backend="sklearn",
    model_type="regressor",
)

exp_dice = Dice(
    data_dice,
    model_dice,
    method="random",
)

dice_pool = int(max(30, 10 * K_RETURN))

print("Generating DiCE candidates")
print("Pool size:", dice_pool)
print("Desired range:", desired_range)
print("Permitted range:", permitted_range)

t0 = time.time()

dice_res = exp_dice.generate_counterfactuals(
    x_factual_df[FEATURES],
    total_CFs=dice_pool,
    desired_range=desired_range,
    features_to_vary=FEATURES,
    permitted_range=permitted_range,
)

dice_runtime = time.time() - t0
method_runtime["DiCE"] = dice_runtime

ex = dice_res.cf_examples_list[0]

if hasattr(ex, "final_cfs_df_sparse") and ex.final_cfs_df_sparse is not None:
    dice_raw = ex.final_cfs_df_sparse.copy()
else:
    dice_raw = ex.final_cfs_df.copy()

print("Raw DiCE candidates:", dice_raw.shape)
display(dice_raw.head())

dice_common_all = prepare_candidate_table(
    candidates=dice_raw,
    method_name="DiCE",
    factual_row=x_factual,
    df_ref=df,
    desired_range=desired_range,
    soft_constraints=query["soft_constraints"],
    model=cb,
)

dice_top = select_top_k(dice_common_all, K_RETURN)
dice_top["generation_sec"] = dice_runtime

print(f"DiCE runtime: {dice_runtime:.3f} seconds")
print("DiCE returned:", len(dice_top))
print("DiCE strictly valid:", int(dice_top["strictly_valid"].sum()))

display(
    dice_top[
        [
            "method",
            "cf_method_score",
            "surrogate_pred",
            "delta_surrogate_minus_method_score",
            TARGET,
            "strictly_valid",
            "n_violations",
            "distance_to_factual",
            "params_soft_review_budget",
            "params_soft_train_fraction",
            "params_soft_corruption_level",
            "params_backbone",
            "params_image_size_key",
            "params_layers_key",
        ]
    ]
)

Generating DiCE candidates
Pool size: 80
Desired range: [0.7, 1.0]
Permitted range: {'params_soft_train_fraction': [0.2, 1.0], 'params_soft_review_budget': [0.005, 0.5], 'params_soft_corruption_level': ['mild', 'none', 'strong']}


100%|██████████| 1/1 [00:00<00:00,  2.79it/s]

Raw DiCE candidates: (80, 14)


,params_backbone,params_batch_size,params_center_crop_key,params_image_size_key,params_layers_key,params_max_patches_per_image,params_pre_trained,params_reduction,params_soft_corruption_level,params_coreset_sampling_ratio,params_num_neighbors,params_soft_review_budget,params_soft_train_fraction,value
0,resnet18,16,0.875,224,l2,512,True,mean,mild,0.029,9,0.430,0.75,0.705024
1,wide_resnet50_2,16,0.875,224,l2,512,True,mean,mild,0.029,9,0.450,0.75,0.720259
2,resnet34,16,0.875,224,l2,512,True,mean,none,0.029,13,0.500,0.75,0.756051
3,resnet18,16,0.875,224,l2,1024,True,mean,mild,0.029,9,0.494,0.75,0.782018
4,resnet18,16,0.875,224,l1_l2_l3,512,True,mean,mild,0.029,9,0.430,0.75,0.704047


DiCE runtime: 0.364 seconds
DiCE returned: 8
DiCE strictly valid: 8


,method,cf_method_score,surrogate_pred,delta_surrogate_minus_method_score,value,strictly_valid,n_violations,distance_to_factual,params_soft_review_budget,params_soft_train_fraction,params_soft_corruption_level,params_backbone,params_image_size_key,params_layers_key
0,DiCE,0.794842,0.794842,-1.140415e-08,0.794842,True,0,2.326506,0.499,0.70,mild,resnet18,224,l2_l3
1,DiCE,0.782018,0.782018,-1.002629e-08,0.782018,True,0,1.452525,0.494,0.75,mild,resnet18,224,l2
2,DiCE,0.777679,0.777679,1.863735e-08,0.777679,True,0,1.456566,0.496,0.75,mild,resnet18,256,l2
3,DiCE,0.773289,0.773289,-1.679311e-08,0.773289,True,0,1.442424,0.489,0.75,mild,resnet18,224,l2
4,DiCE,0.768456,0.768456,1.041202e-08,0.768456,True,0,0.452525,0.494,0.75,mild,resnet18,224,l2
5,DiCE,0.766264,0.766264,-4.980966e-09,0.766264,True,0,1.301843,0.485,0.52,mild,resnet18,224,l2
6,DiCE,0.762813,0.762813,2.909277e-08,0.762813,True,0,2.436364,0.486,0.75,none,resnet18,256,l2
7,DiCE,0.760298,0.760298,2.290891e-08,0.760298,True,0,1.796064,0.478,0.67,mild,resnet34,224,l2


In [10]:
# ============================================================
# 10) Method 2: CFNOW over observed HPO configuration log
# ============================================================

def generate_cfnow_from_hpo_log(
    df_ref,
    factual_row,
    desired_range,
    soft_constraints,
    model,
):
    """
    CFNOW-style configuration search.

    Uses the observed HPO log as the candidate space:
    - every candidate corresponds to an observed configuration;
    - cf_method_score is the observed HPO-log target value;
    - surrogate_pred is recomputed freshly with the final CatBoost surrogate;
    - candidates are validated with the same objective and soft constraints;
    - candidates are ranked with the same ranking function used for all methods.
    """

    t0 = time.time()

    candidates = df_ref[FEATURES + [TARGET]].copy()
    candidates["cf_method_score"] = pd.to_numeric(candidates[TARGET], errors="coerce")

    for c in CATEGORICAL:
        candidates[c] = candidates[c].astype(str)

    for c in NUMERIC:
        candidates[c] = pd.to_numeric(candidates[c], errors="coerce")

    candidates = candidates.dropna(subset=FEATURES).reset_index(drop=True)

    common = prepare_candidate_table(
        candidates=candidates,
        method_name="CFNOW",
        factual_row=factual_row,
        df_ref=df_ref,
        desired_range=desired_range,
        soft_constraints=soft_constraints,
        model=model,
    )

    runtime = time.time() - t0

    if not common.empty:
        common["generation_sec"] = runtime

    return common, runtime


cfnow_common_all, cfnow_runtime = generate_cfnow_from_hpo_log(
    df_ref=df,
    factual_row=x_factual,
    desired_range=desired_range,
    soft_constraints=query["soft_constraints"],
    model=cb,
)

method_runtime["CFNOW"] = cfnow_runtime

cfnow_top = select_top_k(cfnow_common_all, K_RETURN)

if not cfnow_top.empty:
    cfnow_top["generation_sec"] = cfnow_runtime

print(f"CFNOW runtime: {cfnow_runtime:.3f} seconds")
print("CFNOW candidate pool:", len(cfnow_common_all))
print("CFNOW returned:", len(cfnow_top))
print("CFNOW strictly valid:", int(cfnow_top["strictly_valid"].sum()) if not cfnow_top.empty else 0)

display_cols = [
    "method",
    "cf_method_score",
    "surrogate_pred",
    "delta_surrogate_minus_method_score",
    TARGET,
    "strictly_valid",
    "n_violations",
    "distance_to_factual",
    "params_soft_review_budget",
    "params_soft_train_fraction",
    "params_soft_corruption_level",
    "params_backbone",
    "params_image_size_key",
    "params_layers_key",
]

display_cols = [c for c in display_cols if c in cfnow_top.columns]
display(cfnow_top[display_cols])

CFNOW runtime: 5.136 seconds
CFNOW candidate pool: 8475
CFNOW returned: 8
CFNOW strictly valid: 8


,method,cf_method_score,surrogate_pred,delta_surrogate_minus_method_score,value,strictly_valid,n_violations,distance_to_factual,params_soft_review_budget,params_soft_train_fraction,params_soft_corruption_level,params_backbone,params_image_size_key,params_layers_key
0,CFNOW,0.819979,0.822756,0.002777,0.822756,True,0,4.865509,0.500,0.77,none,wide_resnet50_2,256,l3
1,CFNOW,0.814026,0.819806,0.005780,0.819806,True,0,5.853009,0.500,0.76,none,wide_resnet50_2,256,l3
2,CFNOW,0.819979,0.818890,-0.001089,0.818890,True,0,5.845509,0.500,0.77,none,wide_resnet50_2,256,l3
3,CFNOW,0.819979,0.818843,-0.001136,0.818843,True,0,5.865408,0.495,0.81,none,wide_resnet50_2,256,l3
4,CFNOW,0.814297,0.811736,-0.002561,0.811736,True,0,5.858425,0.495,0.88,none,resnet50,256,l2_l3
5,CFNOW,0.814297,0.809968,-0.004329,0.809968,True,0,6.078009,0.500,0.94,none,wide_resnet50_2,256,l2_l3
6,CFNOW,0.802663,0.809557,0.006895,0.809557,True,0,4.852908,0.495,0.80,none,wide_resnet50_2,256,l2_l3
7,CFNOW,0.814297,0.808299,-0.005998,0.808299,True,0,5.982908,0.495,0.92,none,wide_resnet50_2,256,l2_l3


In [11]:
# ============================================================
# 12) Method 3: ACFX numeric/actionable adapter, top-k version
# with early stopping
# ============================================================

HAS_ACFX = False
ACFX_IMPORT_ERROR = None

try:
    from acfx import AcfxEBM
    from interpret.glassbox import ExplainableBoostingClassifier

    HAS_ACFX = True
    print("ACFX and interpret are available in this kernel.")

except Exception as exc:
    HAS_ACFX = False
    ACFX_IMPORT_ERROR = exc

    print("ACFX is not available in this kernel.")
    print("Import error:", exc)


def snap_numeric_candidate(row, df_ref):
    """
    Snap ACFX numeric outputs to valid search-grid values.
    """
    row = row.copy()

    if "params_soft_review_budget" in row:
        row["params_soft_review_budget"] = snap_budget(float(row["params_soft_review_budget"]))

    if "params_soft_train_fraction" in row:
        v = float(row["params_soft_train_fraction"])
        v = max(SOFT_TRAIN_FRACTION_RANGE[0], min(SOFT_TRAIN_FRACTION_RANGE[1], v))
        row["params_soft_train_fraction"] = round(v, 2)

    if "params_num_neighbors" in row:
        observed = sorted(
            df_ref["params_num_neighbors"]
            .dropna()
            .astype(float)
            .unique()
            .tolist()
        )
        row["params_num_neighbors"] = snap_to_grid(float(row["params_num_neighbors"]), observed)

    if "params_coreset_sampling_ratio" in row:
        observed = sorted(
            df_ref["params_coreset_sampling_ratio"]
            .dropna()
            .astype(float)
            .unique()
            .tolist()
        )
        row["params_coreset_sampling_ratio"] = round(
            snap_to_grid(float(row["params_coreset_sampling_ratio"]), observed),
            3,
        )

    return row


def parse_acfx_output(cf_raw):
    """
    Best-effort parser for possible ACFX output formats.
    """
    if cf_raw is None:
        return None

    if isinstance(cf_raw, pd.DataFrame):
        return cf_raw.iloc[0].to_numpy(dtype=float)

    if isinstance(cf_raw, pd.Series):
        return cf_raw.to_numpy(dtype=float)

    if isinstance(cf_raw, dict):
        for key in ["counterfactual", "cf", "x_cf", "solution"]:
            if key in cf_raw:
                return np.asarray(cf_raw[key], dtype=float).reshape(-1)

    if isinstance(cf_raw, (list, tuple)):
        for item in cf_raw:
            try:
                arr = np.asarray(item, dtype=float).reshape(-1)
                if arr.size > 0:
                    return arr
            except Exception:
                pass

    try:
        return np.asarray(cf_raw, dtype=float).reshape(-1)
    except Exception:
        return None


def make_acfx_bounds(X_acfx, acfx_features, soft_constraints):
    """
    Build ACFX bounds using both observed feature ranges and user soft constraints.
    """
    pbounds = {}

    for col in acfx_features:
        col_min = float(X_acfx[col].min())
        col_max = float(X_acfx[col].max())

        spec = soft_constraints.get(col, {"mode": "free"})
        mode = spec.get("mode", "free")

        if mode == "range":
            lo, hi = spec["bounds"]
            lo = max(col_min, float(lo))
            hi = min(col_max, float(hi))
            pbounds[col] = (lo, hi)

        elif mode == "fixed":
            v = float(spec["value"])
            pbounds[col] = (v, v)

        else:
            pbounds[col] = (col_min, col_max)

    return pbounds


def clamp_candidate_to_soft_constraints(candidate, soft_constraints):
    """
    Ensure ACFX numeric outputs do not violate user-provided numeric soft constraints.
    """
    candidate = candidate.copy()

    for col, spec in soft_constraints.items():
        if col not in candidate:
            continue

        mode = spec.get("mode", "free")

        if col in NUMERIC and mode == "range":
            lo, hi = spec["bounds"]
            candidate[col] = min(max(float(candidate[col]), float(lo)), float(hi))

        elif col in NUMERIC and mode == "fixed":
            candidate[col] = float(spec["value"])

    return candidate


def generate_acfx_numeric_counterfactuals_k(
    df_ref,
    factual_row,
    desired_range,
    soft_constraints,
    model,
    k=8,
    n_attempts=40,
    perturb_scale=0.15,
    max_time_sec=90,
    min_attempts_before_unique_stop=20,
):
    """
    Multi-candidate ACFX adapter with early stopping.

    ACFX produces one counterfactual per call, so we call it multiple times from
    slightly perturbed numeric query instances. All outputs are converted to full
    HPO configurations, validated with the CatBoost surrogate, deduplicated, and
    ranked using the same wrapper as DiCE and CFNOW.

    Early stopping conditions:
    1. stop if k strictly valid candidates are found;
    2. stop if k unique candidates are found after a minimum number of attempts;
    3. stop if max_time_sec is reached;
    4. stop after n_attempts.
    """

    global HAS_ACFX, ACFX_IMPORT_ERROR

    if not HAS_ACFX:
        print("Skipping ACFX because the package is unavailable in this kernel.")
        if ACFX_IMPORT_ERROR is not None:
            print("ACFX import error:", ACFX_IMPORT_ERROR)
        return pd.DataFrame(), np.nan

    t0 = time.time()

    acfx_features = NUMERIC.copy()

    X_acfx = df_ref[acfx_features].copy()

    for c in acfx_features:
        X_acfx[c] = pd.to_numeric(X_acfx[c], errors="coerce")

    valid_mask = X_acfx.notna().all(axis=1)
    X_acfx = X_acfx.loc[valid_mask].reset_index(drop=True)
    df_valid = df_ref.loc[valid_mask].reset_index(drop=True).copy()

    y_sur = predict_surrogate(model, df_valid[FEATURES])

    L, U = desired_range
    y_acfx = ((y_sur >= L) & (y_sur <= U)).astype(int)

    if len(np.unique(y_acfx)) < 2:
        print(
            "Skipping ACFX: derived success target has only one class. "
            "Try a less strict or broader objective."
        )
        return pd.DataFrame(), time.time() - t0

    ebm = ExplainableBoostingClassifier(
        random_state=SEED,
        interactions=0,
    )

    ebm.fit(X_acfx, y_acfx)

    pbounds = make_acfx_bounds(
        X_acfx=X_acfx,
        acfx_features=acfx_features,
        soft_constraints=soft_constraints,
    )

    adjacency_matrix = np.zeros((len(acfx_features), len(acfx_features)), dtype=float)
    causal_order = list(range(len(acfx_features)))

    explainer = AcfxEBM(ebm)

    explainer.fit(
        X=X_acfx,
        adjacency_matrix=adjacency_matrix,
        causal_order=causal_order,
        pbounds=pbounds,
        y=y_acfx,
        features_order=acfx_features,
    )

    base_query = np.array(
        [float(factual_row[c]) for c in acfx_features],
        dtype=float,
    )

    lows = np.array([pbounds[c][0] for c in acfx_features], dtype=float)
    highs = np.array([pbounds[c][1] for c in acfx_features], dtype=float)
    spans = np.maximum(highs - lows, 1e-9)

    query_instances = []

    # First attempt: exact factual numeric values clipped to ACFX bounds.
    query_instances.append(np.clip(base_query, lows, highs))

    # Additional attempts: perturbed numeric starting points inside bounds.
    rng_local = np.random.default_rng(SEED)

    for _ in range(n_attempts - 1):
        noise = rng_local.normal(
            loc=0.0,
            scale=perturb_scale,
            size=len(acfx_features),
        ) * spans

        qi = np.clip(base_query + noise, lows, highs)
        query_instances.append(qi)

    candidate_rows = []
    seen_candidates = set()

    for attempt_i, qi in enumerate(query_instances, start=1):
        elapsed = time.time() - t0

        if elapsed > max_time_sec:
            print(f"Stopping ACFX after {max_time_sec} seconds.")
            break

        try:
            cf_raw = explainer.counterfactual(
                desired_class=1,
                query_instance=qi,
            )

            cf_arr = parse_acfx_output(cf_raw)

            if cf_arr is None or len(cf_arr) != len(acfx_features):
                continue

            candidate = factual_row[FEATURES].copy()

            for c, v in zip(acfx_features, cf_arr):
                candidate[c] = float(v)

            candidate = clamp_candidate_to_soft_constraints(candidate, soft_constraints)
            candidate = snap_numeric_candidate(candidate, df_ref)

            key = tuple(str(candidate[c]) for c in FEATURES)

            if key in seen_candidates:
                continue

            seen_candidates.add(key)
            candidate_rows.append(candidate)

            candidate_df_tmp = pd.DataFrame(candidate_rows)

            common_tmp = prepare_candidate_table(
                candidates=candidate_df_tmp,
                method_name="ACFX",
                factual_row=factual_row,
                df_ref=df_ref,
                desired_range=desired_range,
                soft_constraints=soft_constraints,
                model=model,
            )

            n_unique = len(common_tmp)
            n_valid = int(common_tmp["strictly_valid"].sum()) if not common_tmp.empty else 0

            print(
                f"ACFX attempt {attempt_i}/{len(query_instances)} | "
                f"unique={n_unique} | valid={n_valid} | "
                f"elapsed={elapsed:.1f}s"
            )

            if n_valid >= k:
                print(f"Stopping early: found {n_valid} valid ACFX candidates.")
                break

            if (
                n_unique >= k
                and attempt_i >= min_attempts_before_unique_stop
            ):
                print(f"Stopping early: found {n_unique} unique ACFX candidates.")
                break

        except Exception as exc:
            # Keep going because some ACFX attempts may fail depending on the
            # perturbed starting point.
            continue

    if not candidate_rows:
        print("ACFX did not return any parseable candidates.")
        return pd.DataFrame(), time.time() - t0

    candidate_df = pd.DataFrame(candidate_rows)

    common = prepare_candidate_table(
        candidates=candidate_df,
        method_name="ACFX",
        factual_row=factual_row,
        df_ref=df_ref,
        desired_range=desired_range,
        soft_constraints=soft_constraints,
        model=model,
    )

    runtime = time.time() - t0

    if not common.empty:
        common["generation_sec"] = runtime

    return common.reset_index(drop=True), runtime


acfx_common_all, acfx_runtime = generate_acfx_numeric_counterfactuals_k(
    df_ref=df,
    factual_row=x_factual,
    desired_range=desired_range,
    soft_constraints=query["soft_constraints"],
    model=cb,
    k=K_RETURN,
    n_attempts=40,
    perturb_scale=0.15,
    max_time_sec=200,
    min_attempts_before_unique_stop=20,
)

method_runtime["ACFX"] = acfx_runtime

acfx_top = select_top_k(acfx_common_all, K_RETURN)

if not acfx_top.empty:
    acfx_top["generation_sec"] = acfx_runtime

print(f"ACFX runtime: {acfx_runtime}")
print("ACFX candidate pool:", len(acfx_common_all))
print("ACFX returned:", len(acfx_top))
print("ACFX strictly valid:", int(acfx_top["strictly_valid"].sum()) if not acfx_top.empty else 0)

if not acfx_top.empty:
    display(
        acfx_top[
            [
                "method",
                TARGET,
                "strictly_valid",
                "n_violations",
                "distance_to_factual",
                "params_soft_review_budget",
                "params_soft_train_fraction",
                "params_soft_corruption_level",
                "params_coreset_sampling_ratio",
                "params_num_neighbors",
            ]
        ]
    )

INFO:interpret.utils._native:EBM lib loading.
INFO:interpret.utils._native:Finding library for Linux, x86_64, bitsize=64, debug=False
INFO:interpret.utils._native:Loading EBM library /home/jovyan/.conda/envs/pdt_hpo_xai_patchcore/lib/python3.11/site-packages/interpret/utils/../root/bld/lib/libebm_linux_x64.so
INFO:interpret.utils._compressed_dataset:Creating native dataset


ACFX and interpret are available in this kernel.


INFO:interpret.glassbox._ebm._boost:Start boosting
INFO:interpret.utils._native:Booster allocation start
INFO:interpret.utils._native:Booster allocation end
INFO:interpret.utils._native:Deallocation boosting start
INFO:interpret.utils._native:Deallocation boosting end
INFO:interpret.utils._compressed_dataset:Creating native dataset
INFO:interpret.glassbox._ebm._boost:Start boosting
INFO:interpret.utils._native:Booster allocation start
INFO:interpret.utils._native:Booster allocation end
INFO:interpret.utils._native:Deallocation boosting start
INFO:interpret.utils._native:Deallocation boosting end


Sampling from model... Already sampled 5 from 1771 possible in data sampling...
ACFX attempt 1/40 | unique=1 | valid=1 | elapsed=2.5s
Sampling from model... Already sampled 5 from 1771 possible in data sampling...
ACFX attempt 2/40 | unique=2 | valid=2 | elapsed=15.0s
Sampling from model... Already sampled 5 from 1771 possible in data sampling...
ACFX attempt 3/40 | unique=3 | valid=3 | elapsed=27.1s
Sampling from model... Already sampled 5 from 1771 possible in data sampling...
ACFX attempt 4/40 | unique=4 | valid=4 | elapsed=39.3s
Sampling from model... Already sampled 5 from 1771 possible in data sampling...
ACFX attempt 5/40 | unique=5 | valid=5 | elapsed=51.5s
Sampling from model... Already sampled 5 from 1771 possible in data sampling...
ACFX attempt 6/40 | unique=6 | valid=6 | elapsed=63.5s
Sampling from model... Already sampled 5 from 1771 possible in data sampling...
ACFX attempt 7/40 | unique=7 | valid=7 | elapsed=75.6s
Sampling from model... Already sampled 5 from 1771 possi

,method,value,strictly_valid,n_violations,distance_to_factual,params_soft_review_budget,params_soft_train_fraction,params_soft_corruption_level,params_coreset_sampling_ratio,params_num_neighbors
0,ACFX,0.794809,True,0,0.945408,0.495,0.89,mild,0.033,1.0
1,ACFX,0.794066,True,0,0.802908,0.495,0.74,mild,0.035,1.0
2,ACFX,0.790701,True,0,1.424976,0.495,0.84,mild,0.101,5.0
3,ACFX,0.790506,True,0,0.730408,0.495,0.75,mild,0.029,1.0
4,ACFX,0.786811,True,0,0.840408,0.495,0.79,mild,0.023,1.0
5,ACFX,0.780857,True,0,0.602994,0.495,0.73,mild,0.027,6.0
6,ACFX,0.753426,True,0,1.185408,0.495,0.89,mild,0.001,1.0
7,ACFX,0.739636,True,0,0.734156,0.480,0.64,mild,0.029,14.0


In [12]:
# ============================================================
# 13) Combine DiCE, CFNOW, and ACFX for comparison
# ============================================================

method_tables = []

if "dice_top" in globals() and not dice_top.empty:
    method_tables.append(dice_top.copy())

if "cfnow_top" in globals() and not cfnow_top.empty:
    method_tables.append(cfnow_top.copy())

if "acfx_top" in globals() and not acfx_top.empty:
    method_tables.append(acfx_top.copy())

if not method_tables:
    raise ValueError("No counterfactuals were generated by any method.")

all_cfs = pd.concat(method_tables, ignore_index=True)

all_cfs = all_cfs.drop_duplicates(
    subset=["method"] + FEATURES,
    keep="first",
).reset_index(drop=True)

all_cfs.insert(0, "global_cf_id", range(len(all_cfs)))
all_cfs.insert(0, "cf_id", all_cfs["global_cf_id"].astype(int))

# Ensure score columns exist even when a method does not provide one directly.
for col in [
    "cf_method_score",
    "surrogate_pred",
    "delta_surrogate_minus_method_score",
    "surrogate_target_value",
]:
    if col not in all_cfs.columns:
        all_cfs[col] = np.nan

display_cols = [
    "cf_id",
    "global_cf_id",
    "method",
    "cf_method_score",
    "surrogate_pred",
    "delta_surrogate_minus_method_score",
    TARGET,
    "strictly_valid",
    "n_violations",
    "distance_to_factual",
    "generation_sec",
    "params_soft_review_budget",
    "params_soft_train_fraction",
    "params_soft_corruption_level",
    "params_backbone",
    "params_image_size_key",
    "params_layers_key",
]

display_cols = [c for c in display_cols if c in all_cfs.columns]

display(
    all_cfs[display_cols]
    .sort_values(
        ["method", "strictly_valid", "surrogate_pred"],
        ascending=[True, False, False],
    )
    .round(6)
)

print("\nScore-column check:")
display(
    all_cfs[
        [
            "cf_id",
            "method",
            "cf_method_score",
            "surrogate_pred",
            "delta_surrogate_minus_method_score",
            TARGET,
        ]
    ].round(6)
)

,cf_id,global_cf_id,method,cf_method_score,surrogate_pred,delta_surrogate_minus_method_score,value,strictly_valid,n_violations,distance_to_factual,generation_sec,params_soft_review_budget,params_soft_train_fraction,params_soft_corruption_level,params_backbone,params_image_size_key,params_layers_key
16,16,16,ACFX,NaN,0.794809,NaN,0.794809,True,0,0.945408,99.836937,0.495,0.89,mild,resnet18,224,l2
17,17,17,ACFX,NaN,0.794066,NaN,0.794066,True,0,0.802908,99.836937,0.495,0.74,mild,resnet18,224,l2
18,18,18,ACFX,NaN,0.790701,NaN,0.790701,True,0,1.424976,99.836937,0.495,0.84,mild,resnet18,224,l2
19,19,19,ACFX,NaN,0.790506,NaN,0.790506,True,0,0.730408,99.836937,0.495,0.75,mild,resnet18,224,l2
20,20,20,ACFX,NaN,0.786811,NaN,0.786811,True,0,0.840408,99.836937,0.495,0.79,mild,resnet18,224,l2
21,21,21,ACFX,NaN,0.780857,NaN,0.780857,True,0,0.602994,99.836937,0.495,0.73,mild,resnet18,224,l2
22,22,22,ACFX,NaN,0.753426,NaN,0.753426,True,0,1.185408,99.836937,0.495,0.89,mild,resnet18,224,l2
23,23,23,ACFX,NaN,0.739636,NaN,0.739636,True,0,0.734156,99.836937,0.480,0.64,mild,resnet18,224,l2
8,8,8,CFNOW,0.819979,0.822756,0.002777,0.822756,True,0,4.865509,5.135822,0.500,0.77,none,wide_resnet50_2,256,l3
9,9,9,CFNOW,0.814026,0.819806,0.005780,0.819806,True,0,5.853009,5.135822,0.500,0.76,none,wide_resnet50_2,256,l3



Score-column check:


,cf_id,method,cf_method_score,surrogate_pred,delta_surrogate_minus_method_score,value
0,0,DiCE,0.794842,0.794842,-0.000000,0.794842
1,1,DiCE,0.782018,0.782018,-0.000000,0.782018
2,2,DiCE,0.777679,0.777679,0.000000,0.777679
3,3,DiCE,0.773289,0.773289,-0.000000,0.773289
4,4,DiCE,0.768456,0.768456,0.000000,0.768456
5,5,DiCE,0.766264,0.766264,-0.000000,0.766264
6,6,DiCE,0.762813,0.762813,0.000000,0.762813
7,7,DiCE,0.760298,0.760298,0.000000,0.760298
8,8,CFNOW,0.819979,0.822756,0.002777,0.822756
9,9,CFNOW,0.814026,0.819806,0.005780,0.819806


In [13]:
# ============================================================
# 14) Method-level CF summary
# ============================================================

method_summary = (
    all_cfs
    .groupby("method")
    .agg(
        returned=("global_cf_id", "count"),
        strictly_valid=("strictly_valid", "sum"),
        mean_cf_method_score=("cf_method_score", "mean"),
        std_cf_method_score=("cf_method_score", "std"),
        mean_surrogate_f1=("surrogate_pred", "mean"),
        std_surrogate_f1=("surrogate_pred", "std"),
        mean_delta_surrogate_minus_method_score=("delta_surrogate_minus_method_score", "mean"),
        mean_target_value=(TARGET, "mean"),
        mean_distance=("distance_to_factual", "mean"),
        mean_violations=("n_violations", "mean"),
        mean_generation_sec=("generation_sec", "mean"),
    )
    .reset_index()
)

method_summary["strictly_valid_pct"] = (
    100 * method_summary["strictly_valid"] / method_summary["returned"]
)

method_summary = method_summary[
    [
        "method",
        "returned",
        "strictly_valid",
        "strictly_valid_pct",
        "mean_cf_method_score",
        "std_cf_method_score",
        "mean_surrogate_f1",
        "std_surrogate_f1",
        "mean_delta_surrogate_minus_method_score",
        "mean_target_value",
        "mean_distance",
        "mean_violations",
        "mean_generation_sec",
    ]
].sort_values("method")

display(method_summary.round(6))

,method,returned,strictly_valid,strictly_valid_pct,mean_cf_method_score,std_cf_method_score,mean_surrogate_f1,std_surrogate_f1,mean_delta_surrogate_minus_method_score,mean_target_value,mean_distance,mean_violations,mean_generation_sec
0,ACFX,8,8,100.0,NaN,NaN,0.778851,0.020750,NaN,0.778851,0.908333,0.0,99.836937
1,CFNOW,8,8,100.0,0.814939,0.005719,0.814982,0.005652,0.000042,0.814982,5.650210,0.0,5.135822
2,DiCE,8,8,100.0,0.773207,0.011409,0.773207,0.011409,0.000000,0.773207,1.583102,0.0,0.364300


In [14]:
# ============================================================
# 15) Export all-method CFs and metadata
# ============================================================

required_export_cols = [
    "cf_id",
    "global_cf_id",
    "method",
    "cf_method_score",
    "surrogate_pred",
    "delta_surrogate_minus_method_score",
    TARGET,
]

missing_required_export_cols = [c for c in required_export_cols if c not in all_cfs.columns]
if missing_required_export_cols:
    raise ValueError(f"Missing required export columns: {missing_required_export_cols}")

all_cfs_path = EXPORT_DIR / "mvtec_all_methods_cfs.csv"
method_summary_path = EXPORT_DIR / "mvtec_all_methods_cfs_summary.csv"

all_cfs.to_csv(all_cfs_path, index=False)
method_summary.to_csv(method_summary_path, index=False)

meta_all = {
    "dataset": "mvtec_4class",
    "methods": sorted(all_cfs["method"].unique().tolist()),
    "target": TARGET,
    "score_semantics": {
        "cf_method_score": (
            "Score returned or preserved from the counterfactual-generation method. "
            "For DiCE this is the DiCE target column. For CFNOW this is the observed HPO-log target. "
            "For ACFX it may be NaN because the adapter does not return a direct F1 score."
        ),
        "surrogate_pred": "Fresh CatBoost surrogate prediction for the exported candidate configuration.",
        TARGET: "Kept equal to surrogate_pred for objective checking and ranking compatibility.",
        "delta_surrogate_minus_method_score": "surrogate_pred - cf_method_score.",
    },
    "features": FEATURES,
    "categorical": CATEGORICAL,
    "numeric": NUMERIC,
    "surrogate_model_path": str(MODEL_PATH),
    "surrogate_metadata_path": str(METADATA_PATH),
    "surrogate_dataset_path": str(CSV_PATH),
    "factual_index": int(query["factual"]["index"]),
    "factual_value_observed": float(query["factual"]["value_observed"]),
    "factual_value_surrogate": float(query["factual"]["value_surrogate"]),
    "objective": query["objective"],
    "soft_constraints": query["soft_constraints"],
    "selection": query["selection"],
    "desired_range": desired_range,
    "permitted_range": permitted_range,
    "method_runtime": {
        k: float(v) if v is not None and not pd.isna(v) else None
        for k, v in method_runtime.items()
    },
    "n_total_returned": int(len(all_cfs)),
    "n_strictly_valid_total": int(all_cfs["strictly_valid"].sum()),
    "n_with_cf_method_score": int(all_cfs["cf_method_score"].notna().sum()),
}

meta_all_path = EXPORT_DIR / "mvtec_all_methods_cfs_metadata.json"

with open(meta_all_path, "w", encoding="utf-8") as f:
    json.dump(meta_all, f, indent=2, ensure_ascii=False)

print("Saved all-method CFs:", all_cfs_path)
print("Saved method summary:", method_summary_path)
print("Saved metadata:", meta_all_path)

print("\nExport score-column check:")
check = pd.read_csv(all_cfs_path)
display(
    check[
        [
            "cf_id",
            "method",
            "cf_method_score",
            "surrogate_pred",
            "delta_surrogate_minus_method_score",
            TARGET,
        ]
    ].round(6)
)

print("\nMethod summary:")
display(method_summary.round(6))

Saved all-method CFs: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals/mvtec_all_methods_cfs.csv
Saved method summary: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals/mvtec_all_methods_cfs_summary.csv
Saved metadata: /home/jovyan/fast/x_peer_pdt_hyperoptxai/hpo_mvtec_4class/surrogate_pkl_cfs_metadata/mvtec_counterfactuals/mvtec_all_methods_cfs_metadata.json

Export score-column check:


,cf_id,method,cf_method_score,surrogate_pred,delta_surrogate_minus_method_score,value
0,0,DiCE,0.794842,0.794842,-0.000000,0.794842
1,1,DiCE,0.782018,0.782018,-0.000000,0.782018
2,2,DiCE,0.777679,0.777679,0.000000,0.777679
3,3,DiCE,0.773289,0.773289,-0.000000,0.773289
4,4,DiCE,0.768456,0.768456,0.000000,0.768456
5,5,DiCE,0.766264,0.766264,-0.000000,0.766264
6,6,DiCE,0.762813,0.762813,0.000000,0.762813
7,7,DiCE,0.760298,0.760298,0.000000,0.760298
8,8,CFNOW,0.819979,0.822756,0.002777,0.822756
9,9,CFNOW,0.814026,0.819806,0.005780,0.819806



Method summary:


,method,returned,strictly_valid,strictly_valid_pct,mean_cf_method_score,std_cf_method_score,mean_surrogate_f1,std_surrogate_f1,mean_delta_surrogate_minus_method_score,mean_target_value,mean_distance,mean_violations,mean_generation_sec
0,ACFX,8,8,100.0,NaN,NaN,0.778851,0.020750,NaN,0.778851,0.908333,0.0,99.836937
1,CFNOW,8,8,100.0,0.814939,0.005719,0.814982,0.005652,0.000042,0.814982,5.650210,0.0,5.135822
2,DiCE,8,8,100.0,0.773207,0.011409,0.773207,0.011409,0.000000,0.773207,1.583102,0.0,0.364300
